# RAG from Scratch — 100% Local

**Retrieval-Augmented Generation (RAG)** = give an LLM the *right* pages of *your* documents at question time, so it answers from evidence instead of memory.

Today we'll chat with the **Stanford AI Index Report 2025** (457 pages) using models running on this laptop. No API keys, no cloud.

```
            INDEXING (once)                                 QUERYING (every question)
 ┌──────┐   ┌────────┐   ┌──────────┐   ┌──────────┐      ┌──────────┐   ┌──────────┐   ┌──────────┐
 │ PDF  │──▶│ chunks │──▶│  embed   │──▶│ ChromaDB │◀─────│ retrieve │──▶│ augment  │──▶│ generate │
 └──────┘   └────────┘   └──────────┘   └──────────┘      └──────────┘   └──────────┘   └──────────┘
```

| Piece | Library / model | Job |
|---|---|---|
| PDF parsing | `pymupdf` | Pull the text out of each page |
| Embeddings | `nomic-embed-text` via **Ollama** | Turn text into vectors (meaning → numbers) |
| Vector store | `chromadb` | Store vectors + text, find nearest neighbors |
| LLM | `gemma3:12b` via **Ollama** | Write the answer |

**Setup (one time):**
```bash
ollama pull gemma3:12b
ollama pull nomic-embed-text
pip install ollama chromadb pymupdf
```

In [ ]:
import ollama
import chromadb
import pymupdf

PDF_PATH    = "ai_index_2025.pdf"
LLM_MODEL   = "gemma3:12b"
EMBED_MODEL = "nomic-embed-text"
DB_PATH     = "chroma_db"   # Chroma saves to this folder, so we only embed once

## Step 0 — The problem: ask the LLM *without* RAG

Gemma 3 was trained before this report came out. Let's ask it a specific question from the report and see what happens.

In [ ]:
QUESTION = "According to the AI Index Report 2025, how much was U.S. private AI investment in 2024, and how did it compare to China?"

def chat(messages):
    # Stream tokens to the screen as Gemma writes them
    for part in ollama.chat(model=LLM_MODEL, messages=messages, stream=True):
        print(part["message"]["content"], end="", flush=True)
    print()

chat([{"role": "user", "content": QUESTION}])

👆 Either a refusal, a hedge, or a confident-sounding **guess**. (In testing, Gemma confidently said *"$148.6 billion, about 4.5× China's $32.9 billion"*. Both numbers are made up.) The model simply doesn't have this information. Fine-tuning would be slow and expensive, and the answer would go stale as soon as the next report comes out.

**RAG's idea:** don't change the model. Look up the relevant passages and **put them in the prompt**.

## Step 1 — Load the document

`pymupdf` gives us the text of every page. We keep the **page number** (as your PDF viewer shows it) with each page so the model can cite its sources later.

In [ ]:
doc = pymupdf.open(PDF_PATH)
pages = [(i + 1, page.get_text()) for i, page in enumerate(doc)]

print(f"{len(pages)} pages, {sum(len(t) for _, t in pages):,} characters")
print("\n--- page 18 (first 500 chars) ---\n")
print(pages[17][1][:500])

## Step 2 — Chunk

We can't stuff 820K characters into every prompt, and one vector for a whole page blurs many topics together. So we split the text into **small, overlapping chunks**.

- **Chunk size** (here ~150 words): too small loses context, too big dilutes the meaning.
- **Overlap** (here 30 words): a fact that straddles a boundary still lands whole in at least one chunk.

> This is the simplest strategy that works. Real systems also split on headings or sentences (semantic chunking).

In [ ]:
CHUNK_WORDS   = 150
OVERLAP_WORDS = 30

def chunk_text(text, size=CHUNK_WORDS, overlap=OVERLAP_WORDS):
    words = text.split()
    step = size - overlap
    return [" ".join(words[i:i + size]) for i in range(0, max(len(words) - overlap, 1), step)]

chunks = []   # list of {"text": ..., "page": ...}
for page_num, text in pages:
    for c in chunk_text(text):
        if len(c.split()) >= 20:          # skip near-empty chunks (chart-only pages, etc.)
            chunks.append({"text": c, "page": page_num})

print(f"{len(chunks)} chunks\n")
print(chunks[100])

Notice the chart labels and page headers mixed into that chunk: **PDF extraction is messy.** Garbage in, garbage out. Cleaning and parsing documents is often the hardest part of a real RAG project.

## Step 3 — Embed

An **embedding model** turns text into a list of numbers (a vector) where **similar meaning means nearby vectors**. That lets us search by *meaning*, not just keywords.

`nomic-embed-text` wants a small prefix that says whether the text is a document or a search query.

In [ ]:
def embed(texts, kind="document"):
    prefix = "search_document: " if kind == "document" else "search_query: "
    return ollama.embed(model=EMBED_MODEL, input=[prefix + t for t in texts])["embeddings"]

vec = embed(["The U.S. led the world in AI investment."])[0]
print(f"{len(vec)} dimensions: {[round(x, 3) for x in vec[:6]]} ...")

**Quick intuition check:** which sentence is closest in meaning to the query? (Higher = more similar.) Notice that there's almost no word overlap. Embeddings capture *meaning*.

In [ ]:
import numpy as np

query = embed(["How much money went into AI startups?"], kind="query")[0]
candidates = [
    "Venture capital funding for artificial intelligence companies hit a record.",
    "The chapter covers trends in robotics benchmarks.",
    "My cat enjoys sleeping in the sun.",
]
for text, v in zip(candidates, embed(candidates)):
    sim = np.dot(query, v) / (np.linalg.norm(query) * np.linalg.norm(v))
    print(f"{sim:.3f}  {text}")

## Step 4 — Store in a vector database (ChromaDB)

Chroma stores each chunk's **vector + original text + metadata** (the page number) and can quickly find the nearest vectors to a query.

`PersistentClient` saves everything to disk. Embedding ~1,100 chunks takes about 30 seconds on a GPU (a few minutes on CPU). After the first run, this cell finishes instantly.

In [ ]:
client = chromadb.PersistentClient(path=DB_PATH)
collection = client.get_or_create_collection("ai_index_2025", configuration={"hnsw": {"space": "cosine"}})

if collection.count() == 0:
    BATCH = 64
    for start in range(0, len(chunks), BATCH):
        batch = chunks[start:start + BATCH]
        collection.add(
            ids=[f"chunk-{start + i}" for i in range(len(batch))],
            documents=[c["text"] for c in batch],
            metadatas=[{"page": c["page"]} for c in batch],
            embeddings=embed([c["text"] for c in batch]),
        )
        print(f"\rembedded {start + len(batch)}/{len(chunks)}", end="")
    print()

print(f"Collection has {collection.count()} chunks")

## Step 5 — Retrieve

Embed the question, then ask Chroma for the **k nearest chunks**. Always look at what came back: **if retrieval is wrong, the answer will be wrong.**

In [ ]:
def retrieve(question, k=5):
    results = collection.query(query_embeddings=embed([question], kind="query"), n_results=k)
    return [
        {"text": doc, "page": meta["page"], "distance": dist}
        for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0])
    ]

for hit in retrieve(QUESTION):
    print(f"[page {hit['page']}]  distance={hit['distance']:.3f}\n   {hit['text'][:200]}...\n")

## Step 6 — Augment + Generate

**Augment:** paste the retrieved chunks into the prompt, labeled with their page numbers.
**Generate:** tell the model to answer **only** from that context, to cite pages, and to admit when the context doesn't contain the answer.

In [ ]:
SYSTEM_PROMPT = (
    "You answer questions using ONLY the provided context from the AI Index Report 2025. "
    "Cite the page for each fact like (p. 18). "
    "If the context does not contain the answer, say you don't know."
)

def ask(question, k=5, show_sources=True):
    hits = retrieve(question, k)
    context = "\n\n".join(f"[page {h['page']}]\n{h['text']}" for h in hits)

    if show_sources:
        print("Sources:", sorted({h["page"] for h in hits}), "\n")

    chat([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ])

ask(QUESTION)

🎉 Same model, same question. Now the answer is specific, correct, and **cited**. You can open the PDF and check it.

## Try it yourself

Some questions to try, plus experiments that show where RAG breaks:

1. A different fact: *"How much did the inference cost of GPT-3.5-level models drop?"*
2. A question **not** in the report. Does it say "I don't know"?
3. Set `k=1`. Does it still find the answer? What about `k=15`?
4. Change `CHUNK_WORDS` to 40 or 600, delete the `chroma_db` folder, and re-run. How do the answers change?

In [ ]:
ask("How many notable AI models did U.S.-based institutions produce in 2024 compared to China and Europe?")

In [ ]:
ask("What was the population of Paris in 1850?")

## Where to go from here

What we built is "naive RAG," and it's the core of every RAG system. Production systems add:

- **Better chunking:** split on document structure, not word counts.
- **Hybrid search:** combine vector search with keyword search (BM25) for names, numbers, and acronyms.
- **Reranking:** retrieve 50 chunks, then use a cross-encoder to keep the best 5.
- **Evaluation:** measure retrieval and answer quality (e.g., **Ragas**) instead of eyeballing them.
- **Agentic RAG (next week):** retrieval becomes a **tool** the agent decides when and how to call. It can rewrite the query, search several times, or skip search entirely.